# 05 · Dashboard Ejecutivo — AI/BI Dashboards

**Objetivo:** diseñar un **dashboard ejecutivo** para CFE a partir de la capa
semántica creada en el lab anterior (`obras_kpis`, `proyectos_kpis`) y del modelo
estrella **gold**. En lugar de construir cada widget manualmente desde cero,
usarás **Genie Code** para generar un plan de dashboard, los datasets SQL y la
lista de visualizaciones sugeridas.

### ¿Qué debe lograr el dashboard?
* Resumir la **inversión total**, el **número de obras**, el **avance promedio** y las **obras en riesgo**
* Permitir análisis por **tipo de obra**, **residencia**, **estado del proyecto** y **proyecto**
* Incluir una página de **detalle de obras** con filtros de negocio
* Servir como insumo para dirección: lectura rápida, KPIs claros y visuales accionables

### Patrón del lab
1. Ejecuta `%run ./_setup`
2. Genera el **prompt dinámico** con tus variables (`CATALOG`, `WORK_SCHEMA`)
3. Pega el prompt en **Genie Code** (`Cmd+I` / `Ctrl+I`)
4. Usa la salida para crear el dashboard en la interfaz de **AI/BI Dashboards**
5. Valida que los datasets y widgets sí respondan las preguntas de negocio

> Este lab no crea el dashboard automáticamente: te guía para diseñarlo de forma consistente
> con tu esquema y con las metric views del lab 04.

In [0]:
%run ./_setup

## 1. Diseño objetivo: páginas, filtros y visualizaciones

### Página 1 — Resumen Ejecutivo
**Preguntas que debe responder:**
* ¿Cuál es la inversión total comprometida?
* ¿Cuántas obras hay en total?
* ¿Cuál es el avance promedio?
* ¿Cuántas obras están en riesgo?
* ¿Cómo se distribuye la inversión por tipo de obra y por residencia?

**Widgets sugeridos:**
* 4 counters: inversión total, total obras, avance promedio, obras en riesgo
* 1 bar chart: inversión por tipo de obra
* 1 bar chart: inversión por residencia regional
* 1 pie chart: distribución por estado del proyecto
* Filtros: tipo de obra, residencia, estado del proyecto

### Página 2 — Análisis por Proyecto
**Preguntas que debe responder:**
* ¿Qué proyectos concentran más inversión?
* ¿Qué cliente y contratista están asociados a esos proyectos?
* ¿Cómo se compara el avance contra la inversión?

**Widgets sugeridos:**
* 1 table: top proyectos por inversión
* 1 bar chart: top 15 proyectos por inversión
* 1 scatter: avance promedio vs inversión por proyecto
* Filtros: cliente, contratista

### Página 3 — Detalle de Obras
**Preguntas que debe responder:**
* ¿Qué obras específicas requieren atención?
* ¿En qué estado, voltaje o situación actual se encuentran?

**Widgets sugeridos:**
* 1 table detallada de obras
* 1 bar chart: obras por situación actual
* Filtros: tipo de obra, voltaje, situación actual, estado

## 2. Datasets recomendados

Usa primero las **metric views** del lab 04 y recurre a tablas gold solo cuando
necesites detalle fila a fila.

| Dataset | Fuente | Uso |
|---|---|---|
| `obras_kpis_resumen` | `obras_kpis` | Counters ejecutivos |
| `obras_por_tipo` | `obras_kpis` | Inversión y avance por tipo de obra |
| `obras_por_residencia` | `obras_kpis` | Inversión por residencia |
| `obras_por_estado_proyecto` | `obras_kpis` | Distribución y total por estado del proyecto |
| `proyectos_kpis` | `proyectos_kpis` | Top proyectos, scatter avance vs inversión |
| `detalle_obras` | `obras_kpis` o `gold_fact_obras` + dimensiones | Tabla de detalle con filtros |
| `obras_por_situacion` | `obras_kpis` | Conteo por situación actual |

### Reglas de diseño
* Prioriza **pocas métricas, muy claras**
* Evita duplicar datasets si una sola consulta sirve para varios widgets
* Usa nombres amigables para ejecutivos
* Alinea filtros entre datasets equivalentes

## 3. Actividad: genera tu dashboard con Genie Code

### Pasos:
1. **Ejecuta la celda de abajo** para obtener el prompt con tus variables.
2. **Copia el prompt** y pégalo en **Genie Code** (`Cmd+I` / `Ctrl+I`).
3. Pide que te entregue:
   * el **plan del dashboard** (páginas + widgets)
   * los **datasets SQL**
   * la **lista de filtros**
   * una **tabla resumen** que mapee widget → dataset → tipo de visualización
4. Revisa que use primero las metric views `obras_kpis` y `proyectos_kpis`.
5. Crea un dashboard nuevo en **AI/BI Dashboards** y pega/adapta los SQL.
6. Agrega las visualizaciones sugeridas y comprueba que los nombres de negocio sean claros.

### Checklist de revisión
* El dashboard tiene **3 páginas**
* Los counters salen de un dataset ejecutivo de una sola fila
* Las agregaciones en metric views usan `MEASURE(...)` y `GROUP BY ALL`
* La página de detalle sí permite filtrar por estado, tipo de obra y situación actual
* No hay widgets redundantes que respondan exactamente lo mismo

In [0]:
dashboard_name = f"Dashboard Ejecutivo CFE - {spark.sql('SELECT current_user()').first()[0].split('@')[0].replace('.', ' ').title()}"

prompt = f"""
Diseña un dashboard ejecutivo de AI/BI para el caso CFE.
Catálogo: {CATALOG} | Esquema: {WORK_SCHEMA}
Nombre sugerido del dashboard: {dashboard_name}

===== OBJETIVO =====
Crear un dashboard para dirección que resuma inversión, volumen y avance de obras de construcción,
con 3 páginas: Resumen Ejecutivo, Análisis por Proyecto y Detalle de Obras.

===== FUENTES DISPONIBLES =====
Metric views:
- {CATALOG}.{WORK_SCHEMA}.obras_kpis
- {CATALOG}.{WORK_SCHEMA}.proyectos_kpis

Tablas gold disponibles:
- {CATALOG}.{WORK_SCHEMA}.gold_fact_obras
- {CATALOG}.{WORK_SCHEMA}.gold_dim_proyectos
- {CATALOG}.{WORK_SCHEMA}.gold_dim_estados
- {CATALOG}.{WORK_SCHEMA}.gold_dim_tipos_obra
- {CATALOG}.{WORK_SCHEMA}.gold_dim_residencias
- {CATALOG}.{WORK_SCHEMA}.gold_dim_estados_proyecto

===== REGLAS IMPORTANTES =====
1. Prioriza las metric views sobre las tablas gold.
2. Si consultas {CATALOG}.{WORK_SCHEMA}.obras_kpis o {CATALOG}.{WORK_SCHEMA}.proyectos_kpis:
   - usa MEASURE(`metrica`)
   - usa GROUP BY ALL
   - no uses SELECT *
3. Solo usa tablas gold para el dataset de detalle si las metric views no son suficientes.
4. Usa nombres de datasets y widgets en español, pensados para usuarios ejecutivos.
5. El diseño debe ser compacto y claro: máximo 8 widgets por página.
6. Reutiliza datasets cuando tenga sentido.

===== DATASETS REQUERIDOS =====
Propón SQL para estos datasets:
1. obras_kpis_resumen
   - columnas: total_importe_contrato, total_obras, avance_promedio_construccion, obras_en_riesgo, obras_completadas
   - fuente esperada: {CATALOG}.{WORK_SCHEMA}.obras_kpis
2. obras_por_tipo
   - columnas: tipo_obra, total_importe_contrato, total_obras, avance_promedio_construccion
   - fuente esperada: {CATALOG}.{WORK_SCHEMA}.obras_kpis
3. obras_por_residencia
   - columnas: residencia, total_importe_contrato, total_obras
   - fuente esperada: {CATALOG}.{WORK_SCHEMA}.obras_kpis
4. obras_por_estado_proyecto
   - columnas: estado_proyecto, total_obras, total_importe_contrato
   - fuente esperada: {CATALOG}.{WORK_SCHEMA}.obras_kpis
5. proyectos_kpis
   - columnas: nombre_proyecto, cliente, contratista, importe_total_proyecto, total_obras_proyecto, avance_promedio_proyecto
   - fuente esperada: {CATALOG}.{WORK_SCHEMA}.proyectos_kpis
6. detalle_obras
   - columnas mínimas: nombre_obra, proyecto, estado, tipo_obra, residencia, voltaje, situacion_actual,
     importe_contrato, avance_construccion, avance_general, fecha_inicio_programada, fecha_terminacion_programada
   - fuente sugerida: {CATALOG}.{WORK_SCHEMA}.obras_kpis si alcanza; si no, construir desde gold_fact_obras + dimensiones
7. obras_por_situacion
   - columnas: situacion_actual, total_obras
   - fuente esperada: {CATALOG}.{WORK_SCHEMA}.obras_kpis

===== ESTRUCTURA DEL DASHBOARD =====
Página 1: Resumen Ejecutivo
- 4 counters: Inversión Total, Total Obras, Avance Promedio, Obras en Riesgo
- 1 bar: Inversión por Tipo de Obra
- 1 bar: Inversión por Residencia Regional
- 1 pie: Distribución por Estado del Proyecto
- filtros: tipo_obra, residencia, estado_proyecto

Página 2: Análisis por Proyecto
- 1 table: Top Proyectos por Inversión
- 1 bar: Top 15 Proyectos por Inversión
- 1 scatter: Avance vs Inversión por Proyecto
- filtros: cliente, contratista

Página 3: Detalle de Obras
- 1 table: Detalle de Obras de Construcción
- 1 bar: Obras por Situación Actual
- filtros: tipo_obra, voltaje, situacion_actual, estado

===== FORMATO DE RESPUESTA =====
Entrégame exactamente 4 secciones:
1. Un resumen del dashboard (objetivo + 3 páginas)
2. Una tabla en markdown con columnas: pagina | widget | tipo_visual | dataset | pregunta_negocio
3. El SQL completo de cada dataset, uno por uno, listo para pegar
4. Recomendaciones finales de diseño (orden de widgets, formato de moneda/porcentaje y filtros)

Usa los nombres fully qualified con {CATALOG}.{WORK_SCHEMA} en todo el SQL.
"""

print(prompt)

## 4. Validar la capa semántica antes de crear el dashboard

Antes de construir el dashboard, comprueba que las metric views del lab 04 responden
las preguntas principales.

In [0]:
%sql
SELECT
  MEASURE(`total_importe_contrato`) AS inversion_total,
  MEASURE(`total_obras`) AS total_obras,
  MEASURE(`avance_promedio_construccion`) AS avance_promedio,
  MEASURE(`obras_en_riesgo`) AS obras_en_riesgo
FROM ${CATALOG}.${WORK_SCHEMA}.obras_kpis
GROUP BY ALL

In [0]:
%sql
SELECT
  `tipo_obra`,
  MEASURE(`total_importe_contrato`) AS importe,
  MEASURE(`total_obras`) AS obras,
  MEASURE(`avance_promedio_construccion`) AS avance_promedio
FROM ${CATALOG}.${WORK_SCHEMA}.obras_kpis
GROUP BY ALL

In [0]:
%sql
SELECT
  `nombre_proyecto`,
  MEASURE(`importe_total_proyecto`) AS importe_total_proyecto,
  MEASURE(`total_obras_proyecto`) AS total_obras_proyecto,
  MEASURE(`avance_promedio_proyecto`) AS avance_promedio_proyecto
FROM ${CATALOG}.${WORK_SCHEMA}.proyectos_kpis
GROUP BY ALL

## 5. Crear el dashboard en la interfaz

1. Ve a **AI/BI Dashboards**.
2. Crea un dashboard nuevo con el nombre sugerido por el prompt.
3. Agrega los datasets SQL generados por Genie Code.
4. Construye las 3 páginas y sus widgets.
5. Publica el dashboard y compártelo con tu equipo.

### Entregable esperado
Un dashboard ejecutivo con:
* KPIs ejecutivos arriba
* análisis por proyecto en la segunda página
* detalle operativo en la tercera página
* filtros consistentes entre widgets relacionados